# AgentCore Memory Branching을 사용하는 Strands Multi-Agent System

## 소개

이 Notebook에서는 AWS AgentCore Memory와 Strands framework를 사용하여 **Memory Branching을 갖춘 Multi-Agent System**을 구현하는 방법을 살펴봅니다. 이 예제는 원래 대화 thread를 유지하면서 Agent가 대화 기록을 fork하고 대체 대화 path를 탐색할 수 있게 하는 고급 Memory 기능인 **Conversation Branching**을 보여 줍니다.

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | Memory Branching을 사용하는 단기 대화                                                         |
| Agent 사용 사례       | Travel Planning Assistant                                                        |
| Agentic Framework   | Strands Agents                                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                   |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, Strands Agents, Memory retrieval via Tool           |
| 예제 난이도  | 초급                                                                         |


학습 내용:

- 대화 기록을 fork하도록 Memory Branching을 구현하는 방법
- 서로 다른 대화 branch에서 작업하는 전문 Agent 생성
- Branch 수명 주기 관리(생성, 초기화, branch 간 전환)
- main 대화와 branch 대화 모두에서 대화 컨텍스트 유지

### 시나리오 배경

이 예제에서는 Memory Branching을 보여 주는 **여행 계획 시스템**을 만듭니다.
1. 모든 대화가 저장되는 **main branch** 대화
2. 항공편 옵션을 탐색하도록 main 대화에서 fork되는 **Flight Agent branch**
3. 호텔 옵션을 탐색하도록 main 대화에서 fork되는 **Hotel Agent branch**
4. 두 Agent 모두 main branch의 공유 대화 기록에 액세스

이 접근 방식은 Memory Branching이 다음 기능을 지원하는 방법을 보여 줍니다.
- **병렬 탐색**: Agent가 main 대화에 영향을 주지 않고 "what-if" 시나리오 탐색
- **컨텍스트 보존**: Branch 대화가 원래 대화 기록에 대한 액세스 유지
- **전문 workflow**: 각 branch가 원래 컨텍스트를 기반으로 자체 대화 path를 진행

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항
- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory에 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델에 대한 액세스

먼저 환경을 설정하고 공유 Memory 리소스를 생성하겠습니다.

## 1단계: 환경 설정
Notebook 실행에 필요한 모든 라이브러리를 가져오고 client를 정의합니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import (
    AgentInitializedEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

Amazon Bedrock 모델 및 AgentCore에 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os

region = os.getenv("AWS_REGION", "us-west-2")
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("agentcore-memory")

## 2단계: 공유 Memory 생성
이 섹션에서는 전문 Agent가 공유할 Memory 리소스를 생성합니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Memory...")
    memory_name = memory_name

    # Memory 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,  # 이 Memory 저장소의 고유 이름
        description="Travel Agent STM",  # 사람이 읽을 수 있는 설명
        strategies=[],  # 단기 메모리에는 별도 Memory strategy를 사용하지 않음
        event_expiry_days=7,  # Memory는 7일 후 만료
        max_wait=300,  # Memory 생성을 기다리는 최대 시간(5분)
        poll_interval=10,  # 10초마다 상태 확인
    )

    # Memory ID 추출 및 출력
    memory_id = memory["id"]
    print(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 처리
    print(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()

    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Branching을 지원하는 Memory 이해

생성한 Memory 리소스는 다음 기능을 제공하는 **Conversation Branching**을 지원합니다.

1. **공유 세션, 여러 Branch**: 모든 Agent가 동일한 `memory_id`, `actor_id`, `session_id`를 사용하지만 서로 다른 `branch_name` 값을 사용
2. **컨텍스트 상속**: Branch를 fork하면 parent branch(일반적으로 "main")의 대화 기록을 상속
3. **독립적 발전**: Fork 후 각 branch가 독립된 자체 대화 thread를 유지
4. **Branch별 검색**: Agent가 자체 branch에서 Memory 검색

이 Branching 접근 방식을 사용하면 전문 Agent가 공유 세션 기록을 기반으로 하면서 자체 대화 컨텍스트를 유지할 수 있습니다.

## 3단계: Branching을 지원하는 Memory Hook Provider 생성

이 단계에서는 **Memory Branching**을 구현하는 사용자 지정 `ShortTermMemoryHook` 클래스를 정의합니다. 이 고급 Hook Provider는 기본 메모리 작업에 branch 관리 기능을 추가합니다.

### 주요 기능:
1. **Branch 관리**: 대화 branch 자동 생성 및 초기화
2. **Memory 검색**: 지정된 branch에서 대화 기록 가져오기
3. **Memory 저장**: 새 대화를 적절한 branch에 저장
4. **Branch Forking**: Main 대화 thread에서 새 branch 생성

### Branching 작동 방식:
- 각 Agent가 `branch_name`을 지정할 수 있음(기본값: "main")
- Main이 아닌 branch는 main 대화에서 자동으로 fork
- Branch는 fork 지점까지의 대화 기록을 상속
- Fork 후 각 branch가 독립된 자체 대화 흐름 유지



In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory import MemorySessionManager


class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_id: str, region_name: str = "us-west-2", branch_name: str = "main"):
        """MemorySessionManager로 훅을 초기화합니다.

        인자:
            memory_id: AgentCore Memory ID
            region_name: 메모리 서비스용 AWS 리전
            branch_name: 이 에이전트 메모리의 브랜치 이름(기본값: "main")
        """
        self.memory_manager = MemorySessionManager(memory_id=memory_id, region_name=region_name)
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # actor/session 조합별 세션 객체 cache
        self._branch_initialized = False  # Branch 생성 여부 추적

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """주어진 행위자와 세션의 MemorySession을 가져오거나 생성합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자

        반환값:
            MemorySession 객체
        """
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(actor_id=actor_id, session_id=session_id)
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """main 브랜치가 아니면서 브랜치가 없으면 초기화합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자
        """
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Branch가 이미 존재하는지 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # Fork할 main branch의 마지막 event 가져오기
                main_events = memory_session.list_events(branch_name="main")
                if main_events:
                    last_event = main_events[-1]
                    # 초기 메시지와 함께 branch 생성
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(
                                f"Starting {self.branch_name} branch",
                                MessageRole.ASSISTANT,
                            )
                        ],
                    )
                    logger.info(f"✅ Created branch: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"Failed to initialize branch {self.branch_name}: {e}", exc_info=True)

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        try:
            # Agent 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # 필요한 경우 branch 초기화(main이 아닌 branch)
            if self.branch_name != "main":
                self._initialize_branch(actor_id, session_id)

            # Memory Session 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 이 branch에서 최근 5개 대화 turn 가져오기
            recent_turns = memory_session.get_last_k_turns(k=5, branch_name=self.branch_name)

            if recent_turns:
                # 대화 기록을 컨텍스트 형식으로 변환
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.content.get("role", "unknown").lower()
                        text = message.content.get("content", {}).get("text", "")
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages)
                    logger.info(f"Loaded context from branch '{self.branch_name}' ({len(context_messages)} messages)")

                    # Agent의 system prompt에 컨텍스트 추가
                    event.agent.system_prompt += (
                        f"\n\nRecent conversation history (from {self.branch_name}):\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )

                    logger.info(
                        f"✅ Loaded {len(recent_turns)} recent conversation turns from branch '{self.branch_name}'"
                    )
            else:
                logger.info(f"No previous conversation history found in branch '{self.branch_name}'")

        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}", exc_info=True)

    def on_message_added(self, event: MessageAddedEvent):
        """대화 턴을 메모리의 적절한 브랜치에 저장합니다."""
        try:
            # Agent 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Memory Session 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 마지막 메시지 가져오기
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("Skipping empty message")
                return

            # 역할 문자열을 MessageRole enum에 매핑
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # 메시지를 적절한 branch에 저장
            if self.branch_name == "main":
                # Main branch - 일반적인 방식으로 turn 추가
                memory_session.add_turns(messages=[ConversationalMessage(content_text, message_role)])
            else:
                # Main이 아닌 branch - 기존 branch에 추가해야 함
                # Branch가 없으면 초기화
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # 이 branch의 최신 event 가져오기
                branch_events = memory_session.list_events(branch_name=self.branch_name)
                if branch_events:
                    # Branch 이름을 지정하여 기존 branch에 추가(rootEventId 제외)
                    memory_session.add_turns(
                        messages=[ConversationalMessage(content_text, message_role)],
                        branch={"name": self.branch_name},
                    )
                else:
                    # _initialize_branch가 작동했다면 발생하지 않아야 하지만 예외적으로 처리
                    logger.warning(f"Branch {self.branch_name} not found after initialization")
                    self._initialize_branch(actor_id, session_id)

            logger.debug(f"✅ Stored message in branch '{self.branch_name}': {role_str}")

        except Exception as e:
            logger.error(f"Failed to store message: {e}", exc_info=True)

    def create_branch(
        self,
        actor_id: str,
        session_id: str,
        root_event_id: str,
        branch_name: str,
        messages: list,
    ):
        """새 대화 브랜치를 생성합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자
            root_event_id: 분기 기준 이벤트 ID
            branch_name: 새 브랜치 이름
            messages: 브랜치에 추가할 ConversationalMessage 객체 목록
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.fork_conversation(root_event_id=root_event_id, branch_name=branch_name, messages=messages)

    def list_branches(self, actor_id: str, session_id: str):
        """세션의 모든 브랜치를 나열합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자

        반환값:
            브랜치 정보 목록
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.list_branches()

    def get_session(self, actor_id: str, session_id: str):
        """직접 액세스할 메모리 세션 객체를 가져옵니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자

        반환값:
            MemorySession 객체
        """
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """메모리 훅을 레지스트리에 등록합니다.

        인자:
            registry: 콜백을 등록할 HookRegistry
        """
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 4단계: Memory Branching을 사용하는 Multi-Agent 아키텍처 생성

이 섹션에서는 서로 **다른 Memory branch**를 사용하는 전문 Agent를 생성하여 Branching 기능을 살펴봅니다.

### Branching 전략:
- **Main Branch**: Coordinator의 대화를 저장하고 기본 대화 thread 역할 수행
- **flight_agent_memory Branch**: 항공편 전용 대화를 위한 별도 branch
- **hotel_agent_memory Branch**: 호텔 전용 대화를 위한 별도 branch

각 전문 Agent는 자체 branch에서 작동하며, 최초 사용 시 main 대화에서 해당 branch가 자동으로 fork됩니다. 이를 통해 다음이 가능해집니다.

- 서로 다른 전문 영역의 독립된 대화 흐름
- 도메인별 컨텍스트 격리
- Main 대화 thread 보존

In [ ]:
# 필요한 구성 요소 가져오기
from strands import Agent, tool

In [ ]:
# 각 전문 Agent에 고유 Actor ID를 생성하되 Session ID는 공유
actor_id = f"travel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"travel/{actor_id}/preferences/"

### Branch Memory를 사용하는 전문 Agent 생성

다음으로 system prompt를 정의하고 서로 다른 Memory branch를 사용하는 Agent를 생성합니다. 동일한 `actor_id`와 `session_id`를 사용하면서 서로 다른 `branch_name` 값으로 격리된 대화 컨텍스트를 만드는 방식에 주목하세요.

In [ ]:
# Hotel Booking 전문 Agent용 system prompt
HOTEL_BOOKING_PROMPT = """You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# Flight Booking 전문 Agent용 system prompt
FLIGHT_BOOKING_PROMPT = """You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

In [ ]:
flight_memory_hooks = None
hotel_memory_hooks = None

### Branch별 Memory를 사용하는 Agent Tool 구현

이제 전문 Agent를 도구로 구현합니다. 각 Agent에는 특정 branch 이름으로 구성된 자체 Memory Hook이 제공됩니다.
- Flight Assistant는 `flight_agent_memory` branch 사용
- Hotel Assistant는 `hotel_agent_memory` branch 사용

이러한 Agent가 호출되면 다음이 수행됩니다.
1. Hook이 branch 존재 여부 확인
2. 없으면 main 대화에서 새 branch fork
3. Agent의 대화를 전용 branch에 저장
4. Agent가 계속 main branch의 컨텍스트에 액세스 가능

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    global flight_memory_hooks
    try:
        if flight_memory_hooks is None:
            # "flight_agent_memory" 이름의 branch를 사용하는 Hook 생성
            flight_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="flight_agent_memory",
            )

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id},
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"


@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    global hotel_memory_hooks
    try:
        if hotel_memory_hooks is None:
            # "hotel_agent_memory" 이름의 branch를 사용하는 Hook 생성
            hotel_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="hotel_agent_memory",
            )

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id},
        )

        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Coordinator Agent 생성(Main Branch)

Coordinator Agent는 **main branch**에서 작동하며 전문 Agent에 작업을 위임합니다. 다음 사항에 주목하세요.

- Flight 또는 Hotel Assistant를 호출하면 해당 Agent가 자체 branch fork
- 각 전문 Agent의 branch는 main 대화의 현재 상태에서 시작

In [ ]:
# Coordinator Agent용 system prompt
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
agent_memory_hooks = ShortTermMemoryHook(
    memory_id=memory_id,
    region_name=region,
)

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    hooks=[agent_memory_hooks],
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
    state={"actor_id": actor_id, "session_id": session_id},
)

#### Multi-Agent System이 준비되었습니다!

## Agent 테스트

여행 계획 시나리오로 Multi-Agent System을 테스트해 보겠습니다.

In [ ]:
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
response = travel_agent("I would only like to focus on the flight at the moment. Direct flight with British Airways")

In [ ]:
print("\n=== Viewing Memory Branches ===")

if flight_memory_hooks or hotel_memory_hooks:
    # Branch를 나열할 임의의 Memory Session 가져오기(모두 동일한 세션을 가리킴)
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # 세션의 모든 branch 나열
        branches = memory_session.list_branches()
        print(f"\n📊 Session has {len(branches)} branches total:")
        for branch in branches:
            print(f"  - Branch: {branch.name}")
            print(f"    └─ Events: {len(memory_session.list_events(branch_name=branch.name))}")
            print(f"    └─ Created: {branch.created}")

        print("\n💡 Each branch represents a different agent's memory:")
        print("  • 'main' = Travel coordinator conversations")
        print("  • 'flight_agent_memory' = Flight assistant conversations")
        print("  • 'hotel_agent_memory' = Hotel assistant conversations")

In [ ]:
print("\n=== Accessing Branch-Specific Events ===")

if flight_memory_hooks or hotel_memory_hooks:
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # Main branch(Coordinator)에서 event 가져오기
        main_events = memory_session.list_events(branch_name="main")
        print(f"\n🌳 Main Branch - Coordinator ({len(main_events)} events):")
        if main_events:
            for event in main_events[-3:]:  # 최근 event 3개 표시
                for payload in event.payload:
                    if "conversational" in payload:
                        role = payload["conversational"]["role"]
                        text = payload["conversational"]["content"]["text"]
                        print(f"  {role}: {text[:100]}...")
        else:
            print("  No events found in main branch")

        # Flight Agent branch에서 event 가져오기
        try:
            flight_branch_events = memory_session.list_events(branch_name="flight_agent_memory")
            print(f"\n✈️  Flight Agent Branch ({len(flight_branch_events)} events):")
            if flight_branch_events:
                print("All flight-related conversations are stored here:")
                for event in flight_branch_events[-3:]:  # 최근 event 3개 표시
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - flight assistant wasn't called yet")
        except Exception as e:
            print(f"  Flight branch not created yet: {e}")

        # Hotel Agent branch에서 event 가져오기
        try:
            hotel_branch_events = memory_session.list_events(branch_name="hotel_agent_memory")
            print(f"\n🏨 Hotel Agent Branch ({len(hotel_branch_events)} events):")
            if hotel_branch_events:
                print("All hotel-related conversations are stored here:")
                for event in hotel_branch_events[-3:]:  # 최근 event 3개 표시
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - hotel assistant wasn't called yet")
        except Exception as e:
            print(f"  Hotel branch not created yet: {e}")

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

1. **Memory Branching 기본 사항**: AgentCore Memory에서 대화 branch를 생성하고 관리하는 방법
2. **Branch별 Agent**: 서로 다른 Memory branch에서 작동하는 전문 Agent를 구현하는 방법
3. **Branch 자동 생성**: 최초 사용 시 main 대화에서 branch가 자동으로 fork되는 방식
4. **Branch 지속성**: 각 branch가 독립된 자체 대화 기록을 유지하는 방식
5. **컨텍스트 상속**: Branch 대화가 fork 지점까지 main branch의 컨텍스트를 상속하는 방식

### Memory Branching의 주요 이점:
- **격리**: 각 Agent가 다른 Agent에 영향을 주지 않고 자체 대화 컨텍스트 유지
- **유연성**: Main thread에 영향을 주지 않고 대체 대화 path 탐색
- **구성**: 도메인별 대화를 별도 branch로 정리
- **지속성**: Branch별 Memory가 Agent 인스턴스 전반에서 유지

이 Memory Branching 아키텍처는 서로 다른 Agent가 분리되어 있지만 관련된 대화 컨텍스트를 유지해야 하는 정교한 Multi-Agent System을 구축하는 강력한 접근 방식을 제공합니다.

## 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
# )